# Phase 8 — ViT-Tiny / DeiT-Tiny: FP32, Distillation, QAT, INT8

**Author:** Rafael
**Dataset:** Tiny-ImageNet-200 (200 classes, 64x64 RGB)
**Scope of this notebook:** `vit_tiny` and `deit_tiny`. `deit_tiny` needs
`DistillationTrainer` (hard-label distillation from a frozen `mobilenetv2`
teacher, H4) instead of the base `Trainer` for its FP32 stage -- the reason
this notebook exists at all (`scripts/train.py`'s generic loop hardcodes
`Trainer`, no hook for a different trainer class). `vit_tiny` is trained
alongside it for a clean, identically-configured undistilled baseline (H4's
comparison point) and because their QAT/INT8 stages are now identical calls
(see the note below) -- keeping both in one notebook avoids reconciling this
notebook's `checkpoints/phase_8_.../` layout with `scripts/train.py`'s
`outputs/{runtime}/{experiment}/...` layout for what would otherwise be a
handoff between the two for `vit_tiny` alone.

**D6 revision, found while building this notebook:** the original plan wanted
`vit_tiny`/`deit_tiny`'s QAT stage to swap in
`torch.ao.nn.quantizable.MultiheadAttention` (`swap_quantizable_mha()`) so its
internal Linears could be individually quantized -- a "strict improvement"
over Swin's coarser `qconfig=None` fallback. **Verified broken** with this
codebase's eager-mode `tq.prepare_qat()`: PyTorch registers
`quantizable.MultiheadAttention` in its `observed_to_quantized_custom_module_class`
mapping, and `prepare_qat()`'s first internal step (`convert()`, which runs
*before* `prepare()` attaches any observers) matches on it and calls
`.from_observed()` immediately -- `AttributeError`. Setting `qconfig=None` on
the swapped module avoids that crash but *also* stops `prepare()`/`convert()`
from recursing into its own children, so its Linears never quantize either --
the swap bought nothing under this constraint. **Resolution:** `vit_tiny`/
`deit_tiny`'s `self_attention` is now excluded the same way Swin's
`ShiftedWindowAttention` already was (`exclude_attention_from_qat`, extended
to also match `nn.MultiheadAttention`), and `models/vit_variants.py`'s
`_QuantizableEncoderBlock` no longer brackets it with Quant/DeQuant stubs
(that bracketing assumed a quantized-input-capable MHA, which stock
`nn.MultiheadAttention` is not). Net effect: `vit_tiny`/`deit_tiny`'s QAT
stage is now just `build_qat_from_model()`, the same call every other Phase 8
model uses -- **no more `swap_quantizable_mha()` call in this notebook.**
`swap_quantizable_mha()` itself is kept in `ml/quantization.py` (its
weight-transfer math is independently correct, verified by
`ml.quantization`'s `demo()`) but is not wired into the QAT path for the
reason above.

The other five Phase 8 models (`swin_pico_w2/w4/w8`, `swin_pico_poolmixer`,
`hybrid_bottleneck_swin`) need no notebook -- they train via:
```bash
python -m scripts.train --experiment phase8 --runtime local   # or --runtime pcad
```

**Before running this notebook for real training, read the markdown cell above
each blocking-issue check below and make sure it passes -- these guard against
silently corrupted QAT/distillation results, not just crashes.**

## 1. Imports & reproducibility

In [ ]:
import json
import os
import random
import sys
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import wandb

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

from ml import (
    DataConfig, TrainerConfig, QATConfig,
    create_imagenet_loaders,
    MODEL_REGISTRY, register_model,
    Trainer, DistillationTrainer, make_qat_callback,
    build_qat, build_comparison_table,
    load_best_model, convert_to_int8,
    find_fuse_groups,
    auto_resume_path, compress_checkpoint,
    create_results_summary, disk_mb, gzip_mb,
    compute_flops, make_run_summary,
)
from configs.loader import load_config

from models.vit_variants import vit_tiny, deit_tiny
from models.baselines import MobileNetV2TV

torch.backends.quantized.engine = "fbgemm"


In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR    = project_root / "checkpoints" / "phase_8_efficient_vit_hybrid_attention_training"
RESULTS_DIR = project_root / "results" / "phase_8_efficient_vit_hybrid_attention_training"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

data_cfg = DataConfig(**load_config("data.yaml"))
data_cfg.seed = SEED

# D3/Task 4 pitfalls: ViT-family training needs AdamW warmup + this project's
# transformer-tuned hyperparameters (lr=5e-4, weight_decay=0.05), not the
# CNN-tuned training.yaml defaults -- matches configs/experiments/phase8.yaml's
# `training:` block so this notebook and the CLI-driven 5 models train under the
# same regime.
fp32_cfg = replace(
    TrainerConfig(**load_config("training.yaml")),
    lr=5e-4, weight_decay=0.05, warmup_epochs=5,
    label_smoothing=0.1, grad_clip_norm=1.0,
)
qat_cfg = QATConfig(**load_config("qat.yaml"))
print(device)
print(fp32_cfg)


## 2. Dataset & loaders

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")
data_cfg.dataset_path = train_path
print("Tiny-ImageNet train path:", train_path)

train_ds, val_ds, train_loader, val_loader = create_imagenet_loaders(data_cfg)

print(f"Train samples: {len(train_ds):,}")
print(f"Val   samples: {len(val_ds):,}")
print(f"Classes:       {data_cfg.num_classes}")
print(f"Batches/epoch: train={len(train_loader)}, val={len(val_loader)}")


## 3. Blocking-issue pre-flight checks

`ideas/PHASE8_PLAN.md`'s BLOCKING ISSUES section requires these resolved
*before* spending any training time. Run this section every time the kernel is
restarted -- it's cheap (seconds) relative to a wasted training run.

**Blocking Issue #1 (attention weight-transfer), revised scope:** `swap_quantizable_mha`
is no longer on the QAT call path (see the D6 revision note in Section 0) --
`vit_tiny`/`deit_tiny`'s `self_attention` now stays plain `nn.MultiheadAttention`,
excluded from QAT via `qconfig=None` (`exclude_attention_from_qat`), same
treatment as Swin's `ShiftedWindowAttention`. The check below therefore verifies
two narrower things instead: (a) `swap_quantizable_mha`'s weight-transfer math is
still independently correct, in case it's ever revisited (e.g. via FX graph-mode
QAT, which *does* support `torch.ao.nn.quantizable.MultiheadAttention` properly);
(b) `exclude_attention_from_qat` actually excludes every `LayerNorm`/
`ShiftedWindowAttention`/`MultiheadAttention` instance, and a real model
containing a bare `nn.MultiheadAttention` survives `prepare_qat_model()` end to
end (a regression test for the exact crash this section's docstring documents).

Note: `swap_quantizable_mha`'s self-check caught a real bug while this notebook
was being built -- `torch.ao.nn.quantizable.MultiheadAttention`'s
`batch_first=True` path has a broken final reshape in `torch==2.5.1` (diverged
from `nn.MultiheadAttention` by ~0.7 max-abs-diff on random input;
`batch_first=False` matched to 0.0). Fixed in `ml/quantization.py`'s
`swap_quantizable_mha` by always constructing the quantizable module
`batch_first=False` and wrapping it in `_BatchFirstMHAWrapper`. A second,
deeper issue (the custom-module/`prepare_qat` ordering crash, and the
Quant/DeQuant-stub mismatch it led to in `models/vit_variants.py`) is what
actually took `swap_quantizable_mha` off the QAT path -- both are described in
full in `ml/quantization.py`'s `exclude_attention_from_qat` docstring and
`models/vit_variants.py`'s `_QuantizableEncoderBlock` docstring.

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "ml.quantization"],
    cwd=project_root, capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
assert result.returncode == 0, "ml.quantization self-check failed -- do not proceed to QAT (Blocking Issue #1)"


**Blocking Issue #4 (teacher checkpoint existence).** `DistillationTrainer`
hard-depends on Phase 1's `mobilenetv2_best.pth` (57.99% top-1, this project's
best Phase 1 result) already existing on disk -- fail fast here with a clear
message rather than a late crash mid-distillation-training-loop.

In [ ]:
TEACHER_CKPT_DIR = project_root / "outputs" / "pcad" / "archive_legacy_phases" / "phase_4_5_large_scale" / "mobilenetv2" / "checkpoints"
TEACHER_CKPT_PATH = TEACHER_CKPT_DIR / "mobilenetv2_best.pth"

assert TEACHER_CKPT_PATH.exists(), (
    f"Teacher checkpoint not found at {TEACHER_CKPT_PATH} -- deit_tiny's "
    "DistillationTrainer cannot run without it (Blocking Issue #4)."
)

teacher = load_best_model("mobilenetv2", MobileNetV2TV, TEACHER_CKPT_DIR, device)
teacher_params = sum(p.numel() for p in teacher.parameters())
print(f"Teacher checkpoint OK: {TEACHER_CKPT_PATH}")
print(f"Teacher params: {teacher_params:,} ({teacher_params / 1e6:.2f}M)")


**Blocking Issue #2 (FLOPs undercount, D7/Task 5).** `fvcore`'s default op
handlers have no entry for `aten::scaled_dot_product_attention` --
`nn.MultiheadAttention` (used by `vit_tiny`/`deit_tiny`) dispatches straight to
that fused op, so without a custom handler `compute_flops()` silently drops
every attention layer's QK^T + softmax*V MACs. `ml/reporting.py`'s
`compute_flops()` now registers `_sdpa_flop_jit` unconditionally (a no-op for
every pre-Phase-8 model, which contains no such op) -- verify it here against
the standard transformer FLOP-counting formula
(`2 * num_heads * seq_len^2 * head_dim` per layer, Kaplan et al. 2020 Sec 2.1)
before trusting any Phase 8 efficiency number in Section 8/`ideas/BEST_MODELS.md`.

In [ ]:
from fvcore.nn import FlopCountAnalysis
from ml.reporting import _sdpa_flop_jit

probe = vit_tiny().eval()
dummy_input = torch.zeros(1, 3, data_cfg.img_size, data_cfg.img_size)
flops_result = compute_flops(probe)
print("compute_flops() (with SDPA handler):", flops_result)

# Same analysis without the custom handler, to isolate exactly what it adds.
analysis_no_handler = FlopCountAnalysis(probe, dummy_input)
analysis_no_handler.unsupported_ops_warnings(False)
analysis_no_handler.uncalled_modules_warnings(False)
macs_no_handler = analysis_no_handler.total()
print("Unsupported ops without the handler (expect aten::scaled_dot_product_attention here):")
print(analysis_no_handler.unsupported_ops())

# Confirm fvcore has nothing left uncounted beyond known-harmless elementwise/norm ops
# once the handler is registered.
analysis_with_handler = FlopCountAnalysis(probe, dummy_input)
analysis_with_handler.unsupported_ops_warnings(False)
analysis_with_handler.uncalled_modules_warnings(False)
analysis_with_handler.set_op_handle("aten::scaled_dot_product_attention", _sdpa_flop_jit)
analysis_with_handler.total()
print("Remaining unsupported ops with the handler (expect only elementwise/gelu/norm, no matmul/sdpa):")
print(analysis_with_handler.unsupported_ops())

# Hand-computed reference: vit_tiny is image_size=64, patch_size=8 -> 64 patches
# + 1 cls token = 65-token sequence; hidden_dim=192, num_heads=3 -> head_dim=64;
# num_layers=6 (D3's config table).
seq_len, num_heads, head_dim, num_layers = 65, 3, 64, 6
hand_computed_attn_macs = 2 * num_heads * seq_len * seq_len * head_dim * num_layers
print(f"Hand-computed attention MACs: {hand_computed_attn_macs:,}")

delta = flops_result["macs"] - macs_no_handler
print(f"compute_flops() delta from adding the SDPA handler: {delta:,}")
assert delta == hand_computed_attn_macs, "SDPA handler MACs don't match the hand-computed formula (Blocking Issue #2)"
print("FLOPs verification: OK, SDPA handler matches the hand-computed formula exactly.")


## 4. Model shape check (Task 1 validation)

`vit_tiny`/`deit_tiny` share the exact same architecture and constructor
(`deit_tiny = vit_tiny` in `models/vit_variants.py` -- only the training loop
differs, H4) -- one shape/param-count check covers both. Sanity bound
0.5M-15M params per Task 1's validation spec (catches an accidental
`hidden_dim`/`num_layers` typo that would otherwise silently build a
100M+-parameter model).

Also inspects `find_fuse_groups()`'s output for the patch-embedding stem
(Task 2 validation) -- `vit_tiny`'s `conv_proj` is a lone `Conv2d` with no
adjacent `BatchNorm2d`, so an empty list here is correct, not a missed fusion
opportunity.

In [ ]:
model = vit_tiny(num_classes=data_cfg.num_classes)
x = torch.randn(2, 3, data_cfg.img_size, data_cfg.img_size)
with torch.no_grad():
    out = model(x)
assert out.shape == (2, data_cfg.num_classes), f"unexpected output shape: {out.shape}"

n_params = sum(p.numel() for p in model.parameters())
assert 0.5e6 <= n_params <= 15e6, f"vit_tiny param count {n_params:,} outside the 0.5M-15M sanity bound"
print(f"vit_tiny/deit_tiny: output shape {tuple(out.shape)} OK, {n_params:,} params ({n_params / 1e6:.2f}M)")

fuse_groups = find_fuse_groups(model)
print("find_fuse_groups() on vit_tiny's patch-embed stem:", fuse_groups)
assert fuse_groups == [], "expected no fusable Conv-BN pairs in vit_tiny's stem -- inspect before trusting fuse_map=[]"

try:
    import torchinfo
    print(torchinfo.summary(model, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0))
except ModuleNotFoundError:
    print("torchinfo not installed in this environment -- skipped (Task 2 pitfall: verify separately before trusting its layer table for attention modules).")

del model


## 5. Model registration

In [ ]:
register_model("vit_tiny", vit_tiny, fuse_map=[], lr=fp32_cfg.lr, weight_decay=fp32_cfg.weight_decay)
register_model("deit_tiny", deit_tiny, fuse_map=[], lr=fp32_cfg.lr, weight_decay=fp32_cfg.weight_decay)

print("Registered:", [n for n in MODEL_REGISTRY if n in ("vit_tiny", "deit_tiny")])


## 6. FP32 training -- `vit_tiny`

Plain classification, standard `Trainer` -- no distillation, this is H1/H4's
undistilled baseline.

In [ ]:
fp32_training_results = {}

name = "vit_tiny"
resume_from = auto_resume_path(SAVE_DIR, name)

run_id = None
if resume_from:
    meta_path = SAVE_DIR / f"{name}_meta.json"
    if meta_path.exists():
        run_id = json.loads(meta_path.read_text()).get("wandb_run_id")

print("=" * 72)
print(f"Training: {name}  lr={fp32_cfg.lr}  epochs={fp32_cfg.epochs}  warmup_epochs={fp32_cfg.warmup_epochs}")
if resume_from:
    print("(Resuming from checkpoint)")
print("=" * 72)

model = MODEL_REGISTRY[name]["ctor"]().to(device)

run = wandb.init(
    project="alexnet-phase8",
    group="fp32-phase8",
    name=f"{name}_fp32",
    id=run_id,
    resume="allow" if run_id else None,
    config={**asdict(fp32_cfg), "arch": name, "phase": "fp32",
            "num_classes": data_cfg.num_classes, "img_size": data_cfg.img_size,
            "dataset": "tiny-imagenet-200"},
    tags=["phase8", "vit", "tiny-imagenet", "fp32"],
    mode="offline",
)

trainer = Trainer(
    model, train_loader, val_loader, fp32_cfg,
    device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    wandb_run=run,
    log_file=SAVE_DIR / f"{name}.log",
)
results = trainer.fit(resume_from=resume_from)
wandb.finish()

fp32_training_results[name] = results
del model
torch.cuda.empty_cache()
print("\nvit_tiny FP32 training complete.")


## 7. FP32 training -- `deit_tiny` (H4: hard-label distillation)

Same architecture and hyperparameters as `vit_tiny`, only the loss changes:
`DistillationTrainer` mixes cross-entropy against the true label with
cross-entropy against the frozen `mobilenetv2` teacher's argmax prediction
(hard distillation, Touvron et al. 2021 Table 4). `teacher` was already loaded
and verified frozen-checkpoint-exists in Section 3.

In [ ]:
name = "deit_tiny"
resume_from = auto_resume_path(SAVE_DIR, name)

run_id = None
if resume_from:
    meta_path = SAVE_DIR / f"{name}_meta.json"
    if meta_path.exists():
        run_id = json.loads(meta_path.read_text()).get("wandb_run_id")

print("=" * 72)
print(f"Distillation training: {name}  lr={fp32_cfg.lr}  epochs={fp32_cfg.epochs}")
if resume_from:
    print("(Resuming from checkpoint)")
print("=" * 72)

model = MODEL_REGISTRY[name]["ctor"]().to(device)

run = wandb.init(
    project="alexnet-phase8",
    group="fp32-phase8",
    name=f"{name}_fp32",
    id=run_id,
    resume="allow" if run_id else None,
    config={**asdict(fp32_cfg), "arch": name, "phase": "fp32", "distillation_alpha": 0.5,
            "teacher": "mobilenetv2",
            "num_classes": data_cfg.num_classes, "img_size": data_cfg.img_size,
            "dataset": "tiny-imagenet-200"},
    tags=["phase8", "vit", "distillation", "tiny-imagenet", "fp32"],
    mode="offline",
)

trainer = DistillationTrainer(
    model, train_loader, val_loader, fp32_cfg,
    device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    wandb_run=run,
    log_file=SAVE_DIR / f"{name}.log",
    teacher=teacher, alpha=0.5,
)
results = trainer.fit(resume_from=resume_from)
wandb.finish()

fp32_training_results[name] = results
del model
torch.cuda.empty_cache()
print("\ndeit_tiny distillation training complete.")


## 8. Quantization-Aware Training (QAT) -- `vit_tiny` & `deit_tiny`

Since the D6 revision (Section 0), this is exactly the same call sequence the
generic `build_qat()` helper uses for every other Phase 8 model:
`load_best_model()` -> `build_qat_from_model()` (which calls
`exclude_attention_from_qat()` internally -- now also excludes
`nn.MultiheadAttention`, so `self_attention` stays FP32) -> `fit()` with the
usual `make_qat_callback`. No `swap_quantizable_mha()` call needed here
anymore -- see Section 0/3 for why.

In [ ]:
qat_train_cfg = replace(
    fp32_cfg,
    epochs=qat_cfg.epochs,
    lr=qat_cfg.lr,
    weight_decay=qat_cfg.weight_decay,
    use_amp=False,  # AMP incompatible with fake-quant observers
    warmup_epochs=0,  # short QAT fine-tune schedule, no warmup needed
)

qat_models = {}
qat_training_results = {}

for name in ("vit_tiny", "deit_tiny"):
    spec = MODEL_REGISTRY[name]
    resume_from = auto_resume_path(SAVE_DIR, f"qat_{name}")

    print("=" * 72)
    print(f"QAT fine-tuning: {name}")
    if resume_from:
        print("(Resuming from checkpoint)")
    print("=" * 72)

    qat_model = build_qat(name, save_dir=SAVE_DIR, device=device)
    cb = make_qat_callback(qat_cfg.freeze_bn_epoch, qat_cfg.disable_observer_epoch)

    run = wandb.init(
        project="alexnet-phase8",
        group="qat-phase8",
        name=f"{name}_qat",
        config={
            **asdict(qat_train_cfg), "arch": name, "phase": "qat",
            "freeze_bn_epoch": qat_cfg.freeze_bn_epoch,
            "disable_observer_epoch": qat_cfg.disable_observer_epoch,
            "num_classes": data_cfg.num_classes, "img_size": data_cfg.img_size,
            "dataset": "tiny-imagenet-200",
        },
        tags=["phase8", "vit", "tiny-imagenet", "qat"],
        mode="offline",
    )

    trainer = Trainer(
        qat_model, train_loader, val_loader, qat_train_cfg,
        device, SAVE_DIR, f"qat_{name}", num_classes=data_cfg.num_classes,
        wandb_run=run,
        epoch_callback=cb,
        log_file=SAVE_DIR / f"qat_{name}.log",
    )
    results = trainer.fit(resume_from=resume_from)
    wandb.finish()

    qat_training_results[name] = results
    qat_models[name] = qat_model.cpu()

    del fp32_model, qat_model
    torch.cuda.empty_cache()

print("\nQAT fine-tuning complete for vit_tiny and deit_tiny.")


## 9. INT8 conversion & CPU evaluation

In [ ]:
val_loader_cpu = torch.utils.data.DataLoader(
    val_ds, batch_size=data_cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False,
)
cpu_device = torch.device("cpu")

int8_models = {name: convert_to_int8(m.cpu().eval()) for name, m in qat_models.items()}

for name, m in int8_models.items():
    torch.save(m.state_dict(), SAVE_DIR / f"{name}.pth")
    compress_checkpoint(SAVE_DIR / f"{name}.pth")
print("INT8 conversion done.")

int8_metrics = {}
for name, m in int8_models.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(
        m.cpu().eval(), val_loader_cpu, val_loader_cpu, dummy_cfg,
        cpu_device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    )
    metrics = trainer.evaluate(topk=(1, 5))
    int8_metrics[name] = metrics
    print(f"{name:16s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")


## 10. Comparison table, FLOPs & summary persistence

Includes the mixed-precision-ratio accounting H3 needs (`INT8 size / FP32 size`
-- expected well above Phase 3's ~0.25 for fully-INT8-convertible CNNs, per
D6/H3's design).

In [ ]:
fp32_metrics = {}
fp32_benchmarks = {}
rows = []

for name in ("vit_tiny", "deit_tiny"):
    m = load_best_model(name, MODEL_REGISTRY[name]["ctor"], SAVE_DIR, device)
    trainer = Trainer(m, train_loader, val_loader, replace(fp32_cfg, use_amp=False),
                       device, SAVE_DIR, name, num_classes=data_cfg.num_classes)
    fp32_metrics[name] = trainer.evaluate(topk=(1, 5))
    fp32_benchmarks[name] = trainer.benchmark(warmup=100)

    flops_results = compute_flops(m)
    n_params = sum(p.numel() for p in m.parameters())
    fp32_size_mb = disk_mb(SAVE_DIR / f"{name}_best.pth")
    int8_size_mb = disk_mb(SAVE_DIR / f"{name}.pth")

    rows.append({
        "model": name, "precision": "FP32",
        "top1_%": fp32_metrics[name]["top1"], "top5_%": fp32_metrics[name]["top5"],
        "loss": fp32_metrics[name]["loss"],
        "params_M": n_params / 1e6, "size_MB": fp32_size_mb,
        "macs": flops_results["macs"], "flops": flops_results["flops"],
    })
    rows.append({
        "model": f"{name}_INT8", "precision": "INT8",
        "top1_%": int8_metrics[name]["top1"], "top5_%": int8_metrics[name]["top5"],
        "loss": int8_metrics[name]["loss"],
        "params_M": n_params / 1e6, "size_MB": int8_size_mb,
        "macs": flops_results["macs"], "flops": flops_results["flops"],
        "int8_fp32_size_ratio": (int8_size_mb / fp32_size_mb) if fp32_size_mb else None,
    })

    del m
    torch.cuda.empty_cache()

df = build_comparison_table(rows)
df.to_csv(RESULTS_DIR / "phase8_vit_deit_comparison.csv", index=False)
df


In [ ]:
create_results_summary(
    results={
        "fp32_metrics": fp32_metrics,
        "int8_metrics": int8_metrics,
        "fp32_training_results": fp32_training_results,
        "qat_training_results": qat_training_results,
    },
    config=asdict(fp32_cfg),
    output_path=RESULTS_DIR / "experiment_summary.json",
)
print("Saved:", RESULTS_DIR / "experiment_summary.json")
print("Saved:", RESULTS_DIR / "phase8_vit_deit_comparison.csv")


## W&B — syncing offline runs

All runs were saved locally with `mode="offline"`. Two run groups are tracked:
- **`fp32-phase8`** — one run per model (`vit_tiny`, `deit_tiny`), FP32/distillation training
- **`qat-phase8`** — one run per model, QAT fine-tuning

When ready, sync to the W&B dashboard from a terminal:

```bash
wandb sync --sync-all
```

## Next steps

- Run `python -m scripts.train --experiment phase8 --runtime pcad` (or `local`)
  for the other five models (`swin_pico_w2/w4/w8`, `swin_pico_poolmixer`,
  `hybrid_bottleneck_swin`) -- no notebook needed, already dry-run clean.
- Task 7's cross-phase analysis notebook
  (`notebooks/phase_8_efficient_vit_hybrid_attention_analysis/phase8_results_analysis.ipynb`)
  needs FP32+INT8 results for all seven models before it's meaningful -- not
  started, blocked on the training runs above actually being executed.